###Install Required Libraries


In [1]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes "torchao>=0.16.0" sacrebleu

###Check GPU Availability & Memory

In [2]:
import torch

if torch.cuda.is_available():
    gpu_info = torch.cuda.get_device_properties(0)
    vram_gb = round(gpu_info.total_memory / (1024 ** 3), 2)
    print(f"Available GPU: {gpu_info.name}")
    print(f"Total GPU Memory: {vram_gb} GB")
else:
    print("No GPU available. Please enable T4 GPU in Runtime -> Change runtime type.")

Available GPU: Tesla T4
Total GPU Memory: 14.56 GB


What GPU is available? NVIDIA Tesla T4 with 14.56 GB of VRAM.  
Why is a GPU useful for LLM fine-tuning? Large Language Models contain hundreds of millions or billions of parameters. Fine-tuning requires massive parallel matrix multiplications during forward propagation, loss calculation, and backward gradient updates. A GPU provides thousands of parallel compute cores and high-bandwidth VRAM, allowing training and half-precision (FP16) or 4-bit (QLoRA) computations to run in seconds rather than hours on a CPU.

###Create Your Custom 30-Pair Dataset

In [3]:
from datasets import Dataset

# Custom domain dataset: ByteForge Computers (30 QA pairs)
shop_qa_data = [
    {"question": "What products does ByteForge Computers sell?", "answer": "ByteForge Computers sells laptops, custom desktop PCs, gaming monitors, mechanical keyboards, mice, graphics cards, and networking accessories."},
    {"question": "Where is ByteForge Computers located?", "answer": "ByteForge Computers is located at No. 45, Galle Road, Bambalapitiya, Colombo 04."},
    {"question": "What are the opening hours of ByteForge Computers?", "answer": "The shop is open from 9:00 AM to 7:00 PM Monday through Saturday, and closed on Sundays."},
    {"question": "How much is the ByteCare Basic laptop service package?", "answer": "The ByteCare Basic laptop service package costs LKR 3,500 and includes internal dust cleaning and thermal paste replacement."},
    {"question": "What is the student discount on accessories?", "answer": "University students receive a 15% discount on keyboards, mice, and headsets on Thursdays by showing a valid student ID."},
    {"question": "How long are component reservations held?", "answer": "Reserved PC components and laptops are held for 48 hours before being released back into regular stock."},
    {"question": "Do you offer custom PC building services?", "answer": "Yes. Custom PC assembly is free when you purchase all main components from ByteForge Computers, or costs LKR 4,500 as a standalone service."},
    {"question": "What payment methods do you accept?", "answer": "We accept cash, Visa and MasterCard debit or credit cards, online bank transfers, and Kokopay installment plans."},
    {"question": "Can I return a faulty hardware component?", "answer": "Yes. Faulty items can be exchanged within 14 days of purchase if returned in original packaging with the invoice."},
    {"question": "How much does a 16GB DDR5 RAM stick cost?", "answer": "A Kingston Fury 16GB DDR5 5600MHz RAM module costs LKR 21,500."},
    {"question": "Do you sell NVMe SSD drives?", "answer": "Yes. We stock 500GB, 1TB, and 2TB PCIe Gen4 NVMe SSDs from Samsung, Lexar, and Kingston."},
    {"question": "How much is a 1TB NVMe SSD at ByteForge Computers?", "answer": "A 1TB PCIe Gen4 NVMe SSD costs LKR 26,000 with a 3-year warranty."},
    {"question": "Do you sell laptop chargers?", "answer": "Yes. We sell original and universal 65W and 100W USB Type-C laptop chargers starting at LKR 6,800."},
    {"question": "Is islandwide delivery available?", "answer": "Yes. We provide insured courier delivery across Sri Lanka within 2 to 3 working days for a flat fee of LKR 850."},
    {"question": "What warranty do you provide on laptops?", "answer": "All brand-new laptops come with a 2-year hardware warranty and a 1-year battery and adapter warranty."},
    {"question": "How much is the ByteForge Starter Gaming PC?", "answer": "The ByteForge Starter Gaming PC costs LKR 245,000 and includes an Intel Core i5 processor, 16GB RAM, 1TB SSD, and an RTX 4060 GPU."},
    {"question": "Do you repair broken laptop screens?", "answer": "Yes. Laptop screen replacements take 24 hours and start at LKR 22,000 depending on the panel resolution and refresh rate."},
    {"question": "Do you sell mechanical keyboards?", "answer": "Yes. Wireless hot-swappable mechanical keyboards are available starting at LKR 12,500."},
    {"question": "Can I upgrade my laptop RAM at your shop?", "answer": "Yes. Our technicians install and test laptop RAM upgrades free of labor charges when you buy the RAM module from us."},
    {"question": "Do you sell gaming monitors?", "answer": "Yes. We stock 24-inch and 27-inch IPS gaming monitors with 144Hz and 180Hz refresh rates."},
    {"question": "How much is a 24-inch 144Hz monitor?", "answer": "An ASUS 24-inch 144Hz IPS gaming monitor costs LKR 54,000."},
    {"question": "Do you sell Raspberry Pi and Arduino boards?", "answer": "Yes. We stock ESP32 microcontrollers, Arduino Uno kits, and Raspberry Pi 5 boards for university IoT projects."},
    {"question": "How much is an ESP32 development board at your shop?", "answer": "An ESP32 Wi-Fi and Bluetooth development board costs LKR 1,950."},
    {"question": "Do you offer cash refunds for change of mind?", "answer": "No. We do not offer cash refunds for change of mind, only exchanges or store credit for unopened items within 7 days."},
    {"question": "How can I reserve a graphics card?", "answer": "You can reserve a graphics card by calling 011-2554433 or messaging our official WhatsApp support line."},
    {"question": "Do you sell UPS power backups?", "answer": "Yes. 650VA and 1200VA line-interactive UPS units are available starting at LKR 14,500."},
    {"question": "Do you install Genuine Windows and Office?", "answer": "Yes. OEM Windows 11 Pro and Microsoft Office licenses can be purchased and activated with any laptop or PC."},
    {"question": "How much is a wireless gaming mouse?", "answer": "Logitech and Razer wireless gaming mice start at LKR 11,000."},
    {"question": "Do you sell HDMI and DisplayPort cables?", "answer": "Yes. Braided 4K HDMI 2.1 and DisplayPort 1.4 cables are available for LKR 2,200 each."},
    {"question": "Is there parking available at ByteForge Computers?", "answer": "Yes. Free customer parking for cars and motorbikes is available right in front of our Bambalapitiya showroom."}
]

print(f"Total QA examples in dataset: {len(shop_qa_data)}")
raw_dataset = Dataset.from_list(shop_qa_data)

Total QA examples in dataset: 30


###Load the Base Model & Format Prompts

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model in FP16 for the Tesla T4 GPU
compute_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=compute_dtype,
    device_map="auto"
)

print(f"Loaded {model_id} with {model.num_parameters():,} parameters.")

# Convert QA pairs into chat template strings
def apply_chat_format(row):
    conversation = [
        {"role": "system", "content": "You are a helpful customer support assistant for ByteForge Computers."},
        {"role": "user", "content": row["question"]},
        {"role": "assistant", "content": row["answer"]}
    ]
    return {"text": tokenizer.apply_chat_template(conversation, tokenize=False)}

train_dataset = raw_dataset.map(apply_chat_format, remove_columns=raw_dataset.column_names)
print("\nSample formatted training prompt:\n", train_dataset[0]["text"])

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
W1126 21:06:09.268000 586 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1126 21:06:09.314000 586 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


... (3 outputs left out)

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-0.5B-Instruct with 494,032,768 parameters.


Map:   0%|          | 0/30 [00:00<?, ? examples/s]


Sample formatted training prompt:
 <|im_start|>system
You are a helpful customer support assistant for ByteForge Computers.<|im_end|>
<|im_start|>user
What products does ByteForge Computers sell?<|im_end|>
<|im_start|>assistant
ByteForge Computers sells laptops, custom desktop PCs, gaming monitors, mechanical keyboards, mice, graphics cards, and networking accessories.<|im_end|>



###Configure LoRA Adapter with PEFT

In [5]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


###Fine-Tune Using SFTTrainer

In [6]:
from trl import SFTTrainer, SFTConfig

adapter_save_path = "./byteforge_lora/final_adapter"

sft_config = SFTConfig(
    output_dir="./byteforge_lora",
    num_train_epochs=15,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=3e-4,
    logging_steps=15,
    save_strategy="no",
    fp16=torch.cuda.is_available(),
    bf16=False,
    gradient_checkpointing=True,
    max_length=512,
    dataset_text_field="text",
    report_to="none",
    seed=42
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer
)

trainer.train()

# Save the trained LoRA adapter weights and tokenizer
trainer.model.save_pretrained(adapter_save_path)
tokenizer.save_pretrained(adapter_save_path)
print("Saved fine-tuned LoRA adapter to:", adapter_save_path)

Adding EOS to train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Step,Training Loss
15,2.600041
30,0.722701
45,0.325204
60,0.133259
75,0.092884
90,0.077690
105,0.063469
120,0.058980


Saved fine-tuned LoRA adapter to: ./byteforge_lora/final_adapter


###Test Original vs. Fine-Tuned Model & Compute BLEU

In [7]:
import gc
import pandas as pd
import sacrebleu
from peft import PeftModel
from transformers import AutoModelForCausalLM

# Free training memory
if "model" in globals():
    del model
if "trainer" in globals():
    del trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Reload clean base model and attach saved LoRA adapter
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=compute_dtype,
    device_map="auto"
)
fine_tuned_model = PeftModel.from_pretrained(base_model, adapter_save_path)

def ask_model(target_model, question):
    messages = [
        {"role": "system", "content": "You are a helpful customer support assistant for ByteForge Computers."},
        {"role": "user", "content": question}
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(target_model.device)
    target_model.eval()

    with torch.no_grad():
        output_ids = target_model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

# Test Questions: First 2 are completely unseen questions; next 3 test specific domain facts for BLEU
evaluation_questions = [
    "Do you sell liquid cooling kits for gaming PCs, or can you assemble a custom PC for me?", # Unseen question
    "I am a university student looking for a new keyboard and a microcontroller for an IoT project. Can I get both at your shop?", # Unseen question
    "How much is the ByteCare Basic laptop service package?",
    "What is the student discount on accessories?",
    "How long are component reservations held?"
]

original_responses = []
finetuned_responses = []

for q in evaluation_questions:
    # 1. Temporarily disable LoRA adapter to get the TRUE Original Model output
    with fine_tuned_model.disable_adapter():
        orig_reply = ask_model(fine_tuned_model, q)

    # 2. Keep LoRA adapter enabled to get the Fine-Tuned Model output
    ft_reply = ask_model(fine_tuned_model, q)

    original_responses.append(orig_reply)
    finetuned_responses.append(ft_reply)

# Display comparison table
comparison_df = pd.DataFrame({
    "Question": evaluation_questions,
    "Original Model": original_responses,
    "Fine-Tuned Model": finetuned_responses
})
display(comparison_df)

# Calculate BLEU Score on the 3 factual reference questions
reference_targets = [
    "The ByteCare Basic laptop service package costs LKR 3,500 and includes internal dust cleaning and thermal paste replacement.",
    "University students receive a 15% discount on keyboards, mice, and headsets on Thursdays by showing a valid student ID.",
    "Reserved PC components and laptops are held for 48 hours before being released back into regular stock."
]

orig_bleu = sacrebleu.corpus_bleu(original_responses[2:], [reference_targets])
ft_bleu = sacrebleu.corpus_bleu(finetuned_responses[2:], [reference_targets])

print(f"\nOriginal Model BLEU Score:   {orig_bleu.score:.2f}")
print(f"Fine-Tuned Model BLEU Score: {ft_bleu.score:.2f}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

,Question,Original Model,Fine-Tuned Model
0,Do you sell liquid cooling kits for gaming PCs...,"As an AI language model, I don't have the capa...",Yes. Custom PC assembly is free when you purch...
1,I am a university student looking for a new ke...,"Yes, we do offer keyboards and microcontroller...",Yes. We stock original and RS-232 serial port ...
2,How much is the ByteCare Basic laptop service ...,The ByteCare Basic laptop service package typi...,The ByteCare Basic laptop service package cost...
3,What is the student discount on accessories?,"As an AI language model, I don't have access t...",University students receive a 15% discount on ...
4,How long are component reservations held?,Component reservations on ByteForge Computers ...,Reserved PC components and laptops are held fo...



Original Model BLEU Score:   3.44
Fine-Tuned Model BLEU Score: 100.00
